# Save Best Model

Trains the models from `biodiversity_frontend_forecasting.ipynb` and saves the best-performing model (by MAE) for both the aggregated and country-level forecasts to `../data/models/` using joblib.

In [ ]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
DATA_PATH = Path("../data/interim/strict_forecasting/lpd_terrestrial_strict_last2020_unitsusable_global_zeroskeep.csv")
MODELS_PATH = Path("../data/models")
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# ── Feature engineering (copied from biodiversity_frontend_forecasting.ipynb) ──

reference_columns = list(df.columns)
year_columns = [col for col in reference_columns if str(col).isdigit()]

static_numeric_features = ["latitude", "longitude"]
static_categorical_features = [
    "class", "family", "ipbes_subregion", "system_group", "t_realm", "t_biome", "units"
]
identifier_columns = ["id", "binomial", "common_name", "location", "country"]

ref_set = set(reference_columns)
static_numeric_features = [c for c in static_numeric_features if c in ref_set]
static_categorical_features = [c for c in static_categorical_features if c in ref_set]

In [ ]:
def wide_to_long(df):
    id_vars = [c for c in df.columns if not str(c).isdigit()]
    year_cols = [c for c in df.columns if str(c).isdigit()]
    long = df.melt(id_vars=id_vars, value_vars=year_cols, var_name="Year", value_name="Population")
    long["Year"] = long["Year"].astype(int)
    long = long.dropna(subset=["Population"])
    long = long.sort_values(["id", "Year"]).reset_index(drop=True)
    return long


def prepare_country_df(long_df, n_lags=4):
    long_df = long_df.copy()
    long_df["series_id_country"] = long_df["id"].astype(str) + "_" + long_df["country"].astype(str)

    agg_cols = ["binomial", "common_name", "country"] + static_categorical_features
    agg_cols = [c for c in agg_cols if c in long_df.columns]

    country_df = (
        long_df.groupby(["series_id_country", "Year"] + agg_cols, as_index=False)["Population"]
        .mean()
    )

    result_rows = []
    for sid, grp in country_df.groupby("series_id_country"):
        grp = grp.sort_values("Year").reset_index(drop=True)
        grp["log_population"] = np.log1p(grp["Population"])
        for lag in range(1, n_lags + 1):
            grp[f"lag_{lag}"] = grp["log_population"].shift(lag)
        grp["year_gap_from_prev"] = grp["Year"].diff().fillna(1)
        grp["rolling_mean_3"] = grp["log_population"].rolling(3, min_periods=1).mean()
        grp["rolling_std_3"] = grp["log_population"].rolling(3, min_periods=1).std().fillna(0)
        grp["population_difference"] = grp["log_population"].diff().fillna(0)
        grp["population_growth_rate"] = grp["log_population"].pct_change().fillna(0).replace([float("inf"), float("-inf")], 0)
        grp["aggregate_type"] = "country"
        result_rows.append(grp)

    return pd.concat(result_rows, ignore_index=True)


def prepare_agg_df(long_df, mode="absolute", n_lags=4):
    long_df = long_df.copy()
    agg_cols = ["binomial", "common_name"] + static_categorical_features
    agg_cols = [c for c in agg_cols if c in long_df.columns]

    n_pop = long_df.groupby(["binomial", "Year"])["Population"].count().reset_index(name="n_populations")

    if mode == "absolute":
        agg = long_df.groupby(["binomial", "Year"] + [c for c in agg_cols if c != "binomial"], as_index=False)["Population"].sum()
        agg["aggregate_type"] = "sum"
    else:
        agg = long_df.groupby(["binomial", "Year"] + [c for c in agg_cols if c != "binomial"], as_index=False)["Population"].mean()
        agg["aggregate_type"] = "mean"

    agg = agg.merge(n_pop, on=["binomial", "Year"], how="left")
    agg["series_id_species"] = agg["binomial"]

    result_rows = []
    for sid, grp in agg.groupby("series_id_species"):
        grp = grp.sort_values("Year").reset_index(drop=True)
        grp["log_population"] = np.log1p(grp["Population"])
        for lag in range(1, n_lags + 1):
            grp[f"lag_{lag}"] = grp["log_population"].shift(lag)
        grp["year_gap_from_prev"] = grp["Year"].diff().fillna(1)
        grp["rolling_mean_3"] = grp["log_population"].rolling(3, min_periods=1).mean()
        grp["rolling_std_3"] = grp["log_population"].rolling(3, min_periods=1).std().fillna(0)
        grp["population_difference"] = grp["log_population"].diff().fillna(0)
        grp["population_growth_rate"] = grp["log_population"].pct_change().fillna(0).replace([float("inf"), float("-inf")], 0)
        result_rows.append(grp)

    return pd.concat(result_rows, ignore_index=True)


long_df = wide_to_long(df)
country_df = prepare_country_df(long_df, n_lags=4)
agg_df = prepare_agg_df(long_df, mode="absolute", n_lags=4)

print("country_df shape:", country_df.shape)
print("agg_df shape:", agg_df.shape)

In [ ]:
# ── Feature columns ──

base_numeric = [
    "Year", "lag_1", "lag_2", "lag_3", "lag_4",
    "year_gap_from_prev", "rolling_mean_3", "rolling_std_3",
    "population_difference", "population_growth_rate",
]
base_categorical = [c for c in static_categorical_features if c in country_df.columns]

country_extra_numeric = [c for c in ["latitude", "longitude"] if c in country_df.columns]
country_extra_categorical = [c for c in ["country"] if c in country_df.columns]

country_numeric_features = base_numeric + country_extra_numeric
country_categorical_features = base_categorical + country_extra_categorical
country_feature_cols = country_numeric_features + country_categorical_features

agg_numeric_features = base_numeric + [c for c in ["latitude", "longitude", "n_populations"] if c in agg_df.columns]
agg_categorical_features = [c for c in static_categorical_features if c in agg_df.columns]
agg_feature_cols = agg_numeric_features + agg_categorical_features

print("Country features:", country_feature_cols)
print("Agg features:", agg_feature_cols)

In [ ]:
# ── Model factory ──

def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_features),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), categorical_features)
        ]
    )


def make_model(numeric_features, categorical_features):
    preprocessor = make_preprocessor(numeric_features, categorical_features)
    return Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=313,
            max_depth=24,
            min_samples_split=10,
            min_samples_leaf=3,
            max_features=0.7,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ])


def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    denom = np.mean(np.abs(y_true))
    nmae = mae / denom if denom > 0 else float("nan")
    wape_denom = np.sum(np.abs(y_true))
    wape = np.sum(np.abs(y_true - y_pred)) / wape_denom if wape_denom > 0 else float("nan")
    smape_denom = np.abs(y_true) + np.abs(y_pred)
    smape = np.mean(2 * np.abs(y_true - y_pred) / np.where(smape_denom == 0, 1, smape_denom))
    return {"MAE": mae, "RMSE": rmse, "NMAE": nmae, "WAPE": wape, "sMAPE": smape}

In [ ]:
# ── Training helpers ──

horizons = [3, 5, 10, 15, 20]


def create_direct_horizon_dataset(df, series_id_col, horizon):
    rows = []
    for sid, grp in df.groupby(series_id_col):
        grp = grp.sort_values("Year").reset_index(drop=True)
        for i in range(len(grp) - 1):
            origin_row = grp.iloc[i]
            future_rows = grp[grp["Year"] >= origin_row["Year"] + horizon]
            if future_rows.empty:
                continue
            target_row = future_rows.iloc[0]
            row = origin_row.to_dict()
            row[f"target_population_t_plus_{horizon}"] = target_row["Population"]
            row[f"target_log_t_plus_{horizon}"] = np.log1p(target_row["Population"])
            rows.append(row)
    return pd.DataFrame(rows)


def split_origin_years_for_conformal(origin_years, min_train_years=6, max_eval_years=3, max_calib_years=3):
    origin_years = sorted(origin_years)
    n_years = len(origin_years)
    if n_years < (min_train_years + 2):
        return None, None, None
    n_eval = max(1, min(max_eval_years, n_years // 6))
    n_calib = max(1, min(max_calib_years, n_years // 6))
    if n_years - n_eval - n_calib < min_train_years:
        overflow = min_train_years - (n_years - n_eval - n_calib)
        if n_eval > 1:
            reduction = min(overflow, n_eval - 1)
            n_eval -= reduction
            overflow -= reduction
        if overflow > 0 and n_calib > 1:
            reduction = min(overflow, n_calib - 1)
            n_calib -= reduction
    if n_years - n_eval - n_calib < min_train_years:
        return None, None, None
    train_years = origin_years[: n_years - n_eval - n_calib]
    calib_years = origin_years[n_years - n_eval - n_calib : n_years - n_eval]
    eval_years = origin_years[n_years - n_eval :]
    return train_years, calib_years, eval_years


def train_direct_models(df, series_id_col, feature_cols, numeric_features, categorical_features):
    horizon_models = {}
    horizon_conformal_scores = {}
    horizon_results = []

    for h in horizons:
        df_h = create_direct_horizon_dataset(df, series_id_col=series_id_col, horizon=h).copy()
        if df_h.empty:
            print(f"Skipping horizon {h}: no data")
            continue

        origin_years = sorted(df_h["Year"].unique())
        train_years, calib_years, eval_years = split_origin_years_for_conformal(origin_years)
        if train_years is None:
            print(f"Skipping horizon {h}: not enough years")
            continue

        train_h = df_h[df_h["Year"].isin(train_years)].copy()
        calib_h = df_h[df_h["Year"].isin(calib_years)].copy()
        eval_h  = df_h[df_h["Year"].isin(eval_years)].copy()

        X_train = train_h.reindex(columns=feature_cols)
        X_calib = calib_h.reindex(columns=feature_cols)
        X_eval  = eval_h.reindex(columns=feature_cols)

        y_train_log = train_h[f"target_log_t_plus_{h}"]
        y_calib_log = calib_h[f"target_log_t_plus_{h}"]
        y_eval      = eval_h[f"target_population_t_plus_{h}"]

        model_for_calibration = make_model(numeric_features, categorical_features)
        model_for_calibration.fit(X_train, y_train_log)

        calib_pred_log = model_for_calibration.predict(X_calib)
        calib_abs_errors_log = np.abs(y_calib_log.to_numpy() - calib_pred_log)

        eval_pred_log = model_for_calibration.predict(X_eval)
        eval_pred = np.clip(np.expm1(eval_pred_log), a_min=0, a_max=None)
        metrics = evaluate(y_eval.to_numpy(), eval_pred)

        final_train_h = pd.concat([train_h, calib_h]).sort_values("Year").reset_index(drop=True)
        X_final   = final_train_h.reindex(columns=feature_cols)
        y_final_log = final_train_h[f"target_log_t_plus_{h}"]

        final_model = make_model(numeric_features, categorical_features)
        final_model.fit(X_final, y_final_log)

        horizon_models[h] = final_model
        horizon_conformal_scores[h] = calib_abs_errors_log

        horizon_results.append({
            "horizon": h,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "NMAE": metrics["NMAE"],
            "WAPE": metrics["WAPE"],
            "sMAPE": metrics["sMAPE"],
        })
        print(f"  horizon {h:>2}: MAE={metrics['MAE']:.2f}  RMSE={metrics['RMSE']:.2f}")

    return horizon_models, horizon_conformal_scores, pd.DataFrame(horizon_results)

In [ ]:
print("Training country-level models...")
country_horizon_models, country_horizon_conformal_scores, country_horizon_results = train_direct_models(
    country_df,
    series_id_col="series_id_country",
    feature_cols=country_feature_cols,
    numeric_features=country_numeric_features,
    categorical_features=country_categorical_features,
)
print()
print("Training aggregated (species-level) models...")
agg_horizon_models, agg_horizon_conformal_scores, agg_horizon_results = train_direct_models(
    agg_df,
    series_id_col="series_id_species",
    feature_cols=agg_feature_cols,
    numeric_features=agg_numeric_features,
    categorical_features=agg_categorical_features,
)

print("\n── Country results ──")
print(country_horizon_results.to_string(index=False))
print("\n── Aggregated results ──")
print(agg_horizon_results.to_string(index=False))

In [ ]:
# ── Save all horizon models with joblib ──

for h, model in country_horizon_models.items():
    path = MODELS_PATH / f"country_model_h{h}.joblib"
    joblib.dump(model, path)
    print(f"Saved: {path}")

for h, model in agg_horizon_models.items():
    path = MODELS_PATH / f"agg_model_h{h}.joblib"
    joblib.dump(model, path)
    print(f"Saved: {path}")


In [ ]:
# ── Smoke-test: reload all models ──

for h in country_horizon_models:
    m = joblib.load(MODELS_PATH / f"country_model_h{h}.joblib")
    assert hasattr(m, "predict"), f"country h{h} broken"

for h in agg_horizon_models:
    m = joblib.load(MODELS_PATH / f"agg_model_h{h}.joblib")
    assert hasattr(m, "predict"), f"agg h{h} broken"

print(f"Saved {len(country_horizon_models)} country models: horizons {sorted(country_horizon_models.keys())}")
print(f"Saved {len(agg_horizon_models)} agg models:     horizons {sorted(agg_horizon_models.keys())}")
print("All good ✅")
